# Capítulo 5 · Actividad: ¿este tumor es benigno?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_05/03_actividad_breast_cancer_logistica.ipynb)

Entrena una **regresión logística** que diga si un tumor de mama es **benigno o maligno**. Haz el mismo proceso que en las actividades del Titanic y de los autos: primero entiende los datos y después elige el modelo. El preprocesamiento lo decides tú.

Cada fila es una muestra. De la imagen del núcleo celular se miden 10 cosas (radio, textura, perímetro, área, suavidad, compacidad, concavidad, puntos cóncavos, simetría y dimensión fractal). Cada una aparece tres veces:

| Grupo de columnas | Qué es |
|---|---|
| `mean ...` | Media de esa medida en la muestra |
| `... error` | Error estándar |
| `worst ...` | Media de los tres valores más grandes |
| `target` | **Target**: `0` = maligno, `1` = benigno |

`target` sale al revés de lo que uno esperaría: **1 es el caso benigno**.

Como mínimo, tu notebook debe incluir:

- La exploración de antes de modelar: forma de la tabla, balance de las dos clases, distribuciones, correlaciones y qué variables separan maligno de benigno.
- Una **regresión logística**. Las columnas no están en la misma escala (el área va por cientos y la suavidad por centésimas): decide si estandarizas.
- Métricas de **clasificación** con validación cruzada y en prueba. Como mínimo: accuracy, matriz de confusión y AUC. La accuracy sola engaña: hay más benignos que malignos.
- Tus **conclusiones**: qué modelo te quedas y por qué. Decir que un tumor maligno es benigno no es el mismo error al revés.

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos: `uv run python scripts/download_data.py`

---
## Configuración inicial y carga de datos

Seguimos el flujo completo del capítulo:

```
EDA (forma, balance, escalas, correlaciones) → separar train/test → decisión de preprocesamiento
    → validación cruzada y optimización → evaluación en test (matriz, ROC/AUC, umbral) → conclusiones
```

Dejamos todos los *imports* al principio y fijamos la semilla (`SEED = 42`) para que la notebook funcione siempre igual con *Restart & Run All*.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import set_config
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    learning_curve,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

# Que scikit-learn devuelva DataFrames con nombres de columna cuando sea posible
set_config(transform_output="pandas")
sns.set_theme(style="whitegrid")

# Semilla fija para reproducibilidad
SEED = 42
np.random.seed(SEED)

# Carpeta data/ en la raíz del repo (ruta relativa buscando pyproject.toml)
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

df = pd.read_csv(DATA_DIR / "breast_cancer.csv")
df.head()

---
## 1. Exploración de datos (EDA)

Antes de elegir preprocesamiento o entrenar modelos, respondemos cuatro preguntas clave:
1. ¿Cuántas filas y columnas hay, y si existen nulos o duplicados?
2. ¿Qué tan balanceadas están las clases (`0 = maligno` vs. `1 = benigno`)?
3. ¿Cómo son las distribuciones, escalas y asimetrías de las variables?
4. ¿Qué variables se correlacionan más con el diagnóstico y separan mejor maligno de benigno?

### 1.1 Forma de la tabla, tipos y calidad de datos

In [ ]:
print(f"Filas y columnas: {df.shape}")
print(f"Valores nulos totales: {df.isna().sum().sum()}")
print(f"Filas duplicadas: {df.duplicated().sum()}")
print()
df.info()

La tabla está limpia: **569 muestras y 31 columnas** (30 variables predictoras numéricas de tipo `float64` + `target` de tipo `int64`), con **0 valores nulos** y **0 filas duplicadas**.

### 1.2 Balance de clases (`0 = maligno`, `1 = benigno`)

Recuerda que en este dataset las etiquetas vienen al revés de lo habitual: **`0` significa maligno** y **`1` significa benigno**. Veamos cuántas muestras hay de cada clase.

In [ ]:
# Copia solo para visualizar con etiquetas legibles sin alterar df
df_eda = df.copy()
etiquetas = {0: "Maligno (0)", 1: "Benigno (1)"}
df_eda["diagnostico"] = df_eda["target"].map(etiquetas)

conteo = df_eda["diagnostico"].value_counts()
porcentaje = df_eda["diagnostico"].value_counts(normalize=True) * 100
resumen_clases = pd.DataFrame({"Muestras": conteo, "Porcentaje (%)": porcentaje.round(2)})
display(resumen_clases)

plt.figure(figsize=(6, 4))
ax = sns.countplot(
    data=df_eda,
    x="diagnostico",
    hue="diagnostico",
    order=["Maligno (0)", "Benigno (1)"],
    palette={"Maligno (0)": "#d95f02", "Benigno (1)": "#1b9e77"},
    legend=False,
)
for p in ax.patches:
    alto = int(p.get_height())
    ax.annotate(
        f"{alto} ({alto / len(df):.1%})",
        (p.get_x() + p.get_width() / 2, alto),
        ha="center",
        va="bottom",
        fontsize=10,
    )
ax.set_ylim(0, conteo.max() * 1.15)
ax.set_title("Balance de clases en el dataset")
ax.set_xlabel("Diagnóstico")
ax.set_ylabel("Número de muestras")
plt.tight_layout()
plt.show()

**¿Por qué aquí la accuracy sola engaña?**
- Hay **357 tumores benignos (62,7 %)** frente a **212 malignos (37,3 %)**.
- Un modelo inútil que respondiera siempre `1 (Benigno)` acertaría el **62,7 %** de las veces sin detectar **ningún** tumor maligno.
- Por eso, además de la accuracy, necesitamos mirar la **matriz de confusión**, el **recall de la clase `0` (maligno)**, el **F1 macro** y el **AUC-ROC**.

### 1.3 Escalas muy diferentes, distribuciones y asimetría

La regresión logística combina linealmente todas las columnas y aplica regularización sobre los pesos. Si una columna mide miles y otra milésimas, conviene comprobarlo antes de modelar.

In [ ]:
X_eda = df.drop(columns="target")
resumen_vars = X_eda.describe().T[["min", "mean", "std", "max"]].copy()
resumen_vars["skew"] = X_eda.skew()

cols_contraste = [
    "mean radius",
    "mean area",
    "worst area",
    "mean smoothness",
    "mean fractal dimension",
    "area error",
    "concavity error",
    "worst concave points",
]
print(f"Columnas con asimetría fuerte (|skew| > 1): {(resumen_vars['skew'].abs() > 1).sum()} de {len(resumen_vars)}")
resumen_vars.loc[cols_contraste].round(4)

In [ ]:
vars_dist = ["mean area", "mean smoothness", "area error", "worst concave points"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.ravel(), vars_dist):
    sns.histplot(
        data=df_eda,
        x=col,
        hue="diagnostico",
        hue_order=["Maligno (0)", "Benigno (1)"],
        palette={"Maligno (0)": "#d95f02", "Benigno (1)": "#1b9e77"},
        kde=True,
        stat="density",
        common_norm=False,
        alpha=0.4,
        ax=ax,
    )
    ax.set_title(f"{col} (skew = {df_eda[col].skew():.2f})")
    ax.set_xlabel(col)
    ax.set_ylabel("Densidad")

plt.suptitle("Distribuciones y diferencia de escalas entre clases", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

Dos hallazgos importantes en estas distribuciones:
1. **Escalas incompatibles:** `worst area` llega hasta `4254` y `mean area` a `2501`, mientras que `mean smoothness` o `mean fractal dimension` están entre `0,05` y `0,16` (una diferencia de más de **10 000 veces**).
2. **Asimetría positiva y outliers:** 22 de 30 variables tienen `|skew| > 1`, especialmente las de error estándar como `area error` (`skew = 5,45`) o `concavity error` (`skew = 5,11`), porque los tumores malignos más agresivos generan colas largas hacia la derecha.

### 1.4 Correlaciones y variables que mejor separan maligno de benigno

Calculamos la correlación de Pearson con `target`. Como **`0` es maligno y `1` es benigno**, una correlación **negativa** fuerte significa que cuanto **mayor** es esa medida del núcleo celular, más probable es que el tumor sea **maligno**.

In [ ]:
corr_target = df.corr()["target"].drop("target")
top_corr = corr_target.reindex(corr_target.abs().sort_values(ascending=False).index)

top12 = top_corr.head(12)
top8_cols = top_corr.head(8).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16.5, 6.2), gridspec_kw={"width_ratios": [1, 1.3]})

# Izquierda: top 12 correlaciones con target
df_top12 = pd.DataFrame({
    "Variable": top12.index,
    "Correlación": top12.values,
    "Signo": ["Negativa (↑ valor → Maligno)" if v < 0 else "Positiva" for v in top12.values],
})
sns.barplot(
    data=df_top12,
    x="Correlación",
    y="Variable",
    hue="Signo",
    palette={"Negativa (↑ valor → Maligno)": "#d95f02", "Positiva": "#1b9e77"},
    legend=False,
    ax=axes[0],
)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlim(-0.85, 0.05)
axes[0].set_xlabel("Correlación con target (↑ valor → Maligno)")
axes[0].set_title("Top 12 variables más correlacionadas con target")

# Derecha: matriz de correlación entre las 8 variables principales y target
sns.heatmap(
    df[top8_cols + ["target"]].corr(),
    annot=True,
    fmt=".2f",
    annot_kws={"size": 8.5},
    cmap="coolwarm",
    center=0,
    square=True,
    ax=axes[1],
)
axes[1].set_title("Correlación entre las 8 variables principales y target")

plt.tight_layout()
plt.show()

El heatmap también muestra algo útil: las variables de tamaño (`radius`, `perimeter`, `area`) están fuertemente correlacionadas entre sí (`0,94` a `1,00`), porque miden aspectos geométricos relacionados del mismo núcleo. Esto se conoce como **multicolinealidad**, y la regularización L2 de la regresión logística ayuda a repartir el peso entre ellas sin inestabilidad.

Veamos ahora con **boxplots** y un **diagrama de dispersión** cómo las variables principales separan los tumores malignos de los benignos:

In [ ]:
top4_cols = top_corr.head(4).index.tolist()
display(df_eda.groupby("diagnostico")[top4_cols].mean().round(4))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), top4_cols):
    sns.boxplot(
        data=df_eda,
        x="diagnostico",
        y=col,
        hue="diagnostico",
        order=["Maligno (0)", "Benigno (1)"],
        palette={"Maligno (0)": "#d95f02", "Benigno (1)": "#1b9e77"},
        legend=False,
        ax=ax,
    )
    ax.set_title(f"{col} según diagnóstico")
    ax.set_xlabel("")

plt.suptitle("Variables que mejor separan tumores malignos y benignos", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df_eda,
    x="worst perimeter",
    y="worst concave points",
    hue="diagnostico",
    hue_order=["Maligno (0)", "Benigno (1)"],
    palette={"Maligno (0)": "#d95f02", "Benigno (1)": "#1b9e77"},
    alpha=0.75,
)
plt.title("Separación entre clases con las dos variables más correlacionadas")
plt.xlabel("worst perimeter (peor perímetro)")
plt.ylabel("worst concave points (peores puntos cóncavos)")
plt.tight_layout()
plt.show()

La separación es muy clara:
- Los tumores **malignos (`0`)** presentan núcleos más grandes e irregulares: en promedio tienen casi el triple de `worst concave points` (`0,182` vs. `0,074`) y mucho mayor `worst perimeter` (`141,4` vs. `87,0`) y `worst radius` (`21,1` vs. `13,4`).
- En el *scatterplot*, las dos clases forman dos nubes casi separables por una frontera lineal, lo cual indica que una **regresión logística** es muy adecuada para este problema.

---
## 2. División Train / Test y decisión de preprocesamiento

### 2.1 Separar Train y Test antes de escalar

Separamos el **80 % para entrenamiento** (`train`) y el **20 % para prueba final** (`test`) **antes** de calcular medias o desviaciones para escalar. Usamos `stratify=y` para mantener exactamente la misma proporción de malignos (~37 %) y benignos (~63 %) en ambos conjuntos y evitar **fuga de datos**.

In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print(f"Train: {X_train.shape} | Proporción benigno (1): {y_train.mean():.2%} | maligno (0): {(y_train == 0).mean():.2%}")
print(f"Test:  {X_test.shape} | Proporción benigno (1): {y_test.mean():.2%} | maligno (0): {(y_test == 0).mean():.2%}")

### 2.2 Decisión de preprocesamiento

A partir del EDA, tomamos dos decisiones razonadas que vamos a comprobar con validación cruzada:

1. **¿Estandarizar? Sí (`StandardScaler` dentro de un `Pipeline`).**
   - En `LogisticRegression`, las variables con números enormes (`worst area` hasta 4254) frente a variables diminutas (`mean smoothness` ~0,10) descompensan el optimizador numérico (que puede incluso **no converger**) y hacen que la regularización L2 penalice de forma desigual según la unidad de medida.
   - Colocar `StandardScaler` dentro de un `Pipeline` garantiza que en cada *fold* de validación cruzada se ajusten media y desviación estándar **únicamente con los datos de entrenamiento de ese fold**.
2. **¿Transformar la asimetría (`np.log1p`)?**
   - Como vimos que 22 columnas tienen asimetría positiva fuerte (`|skew| > 1`) y ningún valor negativo (`min >= 0`), podemos probar `FunctionTransformer(np.log1p)` antes de `StandardScaler` y verificar con validación cruzada si realmente mejora a `StandardScaler` solo o si no hace falta complicar el modelo.

---
## 3. Modelado y Validación Cruzada

Usamos `StratifiedKFold` con **5 folds** para estimar el rendimiento de forma estable sin tocar `X_test`. Evaluamos varias métricas a la vez:
- `accuracy`: proporción global de aciertos.
- `f1_macro`: promedio del F1 de malignos y benignos dando el mismo peso a ambas clases (evita que la clase mayoritaria oculte errores).
- `roc_auc`: capacidad del modelo para ordenar correctamente tumores malignos frente a benignos en todos los umbrales.
- `recall_maligno`: proporción de tumores **realmente malignos (`0`)** que el modelo logra detectar.

### 3.1 Comparación de preprocesamientos: sin escalar vs. estandarizado vs. log1p + estandarizado

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

metricas_cv = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "roc_auc": "roc_auc",
    "recall_maligno": make_scorer(recall_score, pos_label=0),
}

candidatos_preproc = {
    "1. Base sin escalar": LogisticRegression(max_iter=1000, random_state=SEED),
    "2. Estandarizado (C=1.0)": Pipeline([
        ("escalador", StandardScaler()),
        ("logistica", LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)),
    ]),
    "3. Log1p + Estandarizado (C=1.0)": Pipeline([
        ("log1p", FunctionTransformer(np.log1p)),
        ("escalador", StandardScaler()),
        ("logistica", LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)),
    ]),
}

filas_cv = []
for nombre, modelo in candidatos_preproc.items():
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always")
        res = cross_validate(modelo, X_train, y_train, cv=cv, scoring=metricas_cv, return_train_score=True)

    hubo_no_conv = any(issubclass(w.category, ConvergenceWarning) for w in avisos)
    if hubo_no_conv:
        print(f"⚠️ {nombre}: apareció ConvergenceWarning (lbfgs no terminó de converger en 1000 iteraciones).")

    filas_cv.append({
        "Modelo": nombre,
        "Train Acc": res["train_accuracy"].mean(),
        "CV Acc (media)": res["test_accuracy"].mean(),
        "CV Acc (desv.)": res["test_accuracy"].std(),
        "CV F1 macro": res["test_f1_macro"].mean(),
        "CV Recall maligno (0)": res["test_recall_maligno"].mean(),
        "CV AUC": res["test_roc_auc"].mean(),
        "¿Convergió?": "No" if hubo_no_conv else "Sí",
    })

tabla_preproc = pd.DataFrame(filas_cv).set_index("Modelo")
tabla_preproc.round(4)

**¿Qué demuestra esta tabla?**
- **El modelo sin escalar falla al converger** (`ConvergenceWarning`) por la diferencia de escalas entre el área y la suavidad, y obtiene el peor rendimiento en validación cruzada (`0,9495` de accuracy y `0,9176` de recall en malignos).
- **Estandarizar con `StandardScaler` resuelve el problema:** converge sin avisos, sube la accuracy en CV a **`0,9780`**, el recall de malignos a **`0,9647`** y el AUC a **`0,9959`**.
- **`Log1p + Estandarizado`** logra resultados prácticamente empatados en CV (`0,9802` de accuracy, `0,9958` de AUC y `0,9706` de recall en malignos). Vamos a mantenerlo para comprobar en test si aporta alguna ventaja real o si basta con `StandardScaler`.

### 3.2 Optimización del hiperparámetro `C` con `GridSearchCV`

En `LogisticRegression`, el hiperparámetro **`C`** es el inverso de la fuerza de regularización:
- **`C` pequeño** (ej. `0,01`, `0,1`) → regularización fuerte, coeficientes más pequeños (modelo más simple, menos sobreajuste, pero si es demasiado pequeño puede subajustar).
- **`C` grande** (ej. `5,0`, `10,0`) → regularización débil, el modelo se ajusta más libremente a los datos de entrenamiento (riesgo de sobreajuste).

Buscamos el mejor `C` con `GridSearchCV` midiendo `accuracy`, `roc_auc` y `recall_maligno` en la misma validación cruzada de 5 folds.

In [ ]:
rejilla_C = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]

pipe_base = Pipeline([
    ("escalador", StandardScaler()),
    ("logistica", LogisticRegression(max_iter=1000, random_state=SEED)),
])

grid = GridSearchCV(
    pipe_base,
    param_grid={"logistica__C": rejilla_C},
    cv=cv,
    scoring=metricas_cv,
    refit="roc_auc",
    return_train_score=True,
)
grid.fit(X_train, y_train)

resultados_grid = pd.DataFrame(grid.cv_results_)[[
    "param_logistica__C",
    "mean_train_accuracy",
    "mean_test_accuracy",
    "mean_test_f1_macro",
    "mean_test_recall_maligno",
    "mean_test_roc_auc",
]].rename(columns={
    "param_logistica__C": "C",
    "mean_train_accuracy": "Train Acc",
    "mean_test_accuracy": "CV Acc",
    "mean_test_f1_macro": "CV F1 macro",
    "mean_test_recall_maligno": "CV Recall maligno (0)",
    "mean_test_roc_auc": "CV AUC",
})

print(f"Mejor C según AUC: {grid.best_params_['logistica__C']} (CV AUC = {grid.best_score_:.4f})")
mejor_c_acc = resultados_grid.loc[resultados_grid["CV Acc"].idxmax(), "C"]
print(f"Mejor C según Accuracy: {mejor_c_acc} (CV Acc = {resultados_grid['CV Acc'].max():.4f})")
display(resultados_grid.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Izquierda: Accuracy train vs validación
axes[0].plot(resultados_grid["C"], resultados_grid["Train Acc"], marker="o", label="Train Acc")
axes[0].plot(resultados_grid["C"], resultados_grid["CV Acc"], marker="s", label="Validación (CV) Acc")
axes[0].set_xscale("log")
axes[0].set_title("Accuracy según regularización C")
axes[0].set_xlabel("C (escala logarítmica)")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

# Derecha: AUC y Recall maligno en validación cruzada
axes[1].plot(resultados_grid["C"], resultados_grid["CV AUC"], marker="o", color="#1b9e77", label="CV AUC")
axes[1].plot(resultados_grid["C"], resultados_grid["CV Recall maligno (0)"], marker="^", color="#d95f02", label="CV Recall maligno (0)")
axes[1].set_xscale("log")
axes[1].set_title("AUC y Recall de malignos según C")
axes[1].set_xlabel("C (escala logarítmica)")
axes[1].set_ylabel("Puntuación")
axes[1].legend()

plt.tight_layout()
plt.show()

Las gráficas cuentan una historia muy clara:
- Si `C` es demasiado pequeño (`0,01`), el modelo queda **demasiado restringido (subajuste)**: tanto Train como CV bajan, y el recall de malignos cae a `0,8706`.
- Si `C` es grande (`5,0` o `10,0`), la accuracy en Train sigue subiendo (`0,9918`), pero en CV empeoran la accuracy, el AUC y el recall de malignos: señal de **inicio de sobreajuste**.
- En la zona óptima aparecen dos candidatos claros: **`C = 0,1`** (máxima accuracy en CV: `0,9824`) y **`C = 1,0`** (máximo AUC en CV: `0,9959` y máximo recall de malignos en CV: `0,9647`).

---
## 4. Evaluación final en el conjunto de prueba (Test)

Ahora entrenamos nuestros cuatro modelos candidatos con **todo `X_train`** y los evaluamos por primera vez sobre **`X_test`** (las 114 muestras reservadas desde el inicio).

### 4.1 Tabla comparativa completa (Validación Cruzada y Test)

In [ ]:
modelos = {
    "1. Base sin escalar (C=1.0)": LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
    "2. Log1p + Estandarizado (C=1.0)": Pipeline([
        ("log1p", FunctionTransformer(np.log1p)),
        ("escalador", StandardScaler()),
        ("logistica", LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)),
    ]),
    "3. Estandarizado · Grid Acc (C=0.1)": Pipeline([
        ("escalador", StandardScaler()),
        ("logistica", LogisticRegression(C=0.1, max_iter=1000, random_state=SEED)),
    ]),
    "4. Estandarizado · Grid AUC (C=1.0)": Pipeline([
        ("escalador", StandardScaler()),
        ("logistica", LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)),
    ]),
}

filas_final = []
for nombre, modelo in modelos.items():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        cv_res = cross_validate(modelo, X_train, y_train, cv=cv, scoring=metricas_cv)
        modelo.fit(X_train, y_train)

    pred_test = modelo.predict(X_test)
    # Probabilidad de la clase 1 (benigno) y clase 0 (maligno)
    prob_benigno = modelo.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, pred_test, labels=[0, 1])

    filas_final.append({
        "Modelo": nombre,
        "Acc CV (media)": cv_res["test_accuracy"].mean(),
        "AUC CV (media)": cv_res["test_roc_auc"].mean(),
        "Acc Test": accuracy_score(y_test, pred_test),
        "F1 macro Test": f1_score(y_test, pred_test, average="macro"),
        "Recall Maligno (0)": recall_score(y_test, pred_test, pos_label=0),
        "Precision Maligno (0)": precision_score(y_test, pred_test, pos_label=0),
        "Recall Benigno (1)": recall_score(y_test, pred_test, pos_label=1),
        "AUC Test": roc_auc_score(y_test, prob_benigno),
        "Falsos benignos (0→1)": int(cm[0, 1]),
        "Falsas alarmas (1→0)": int(cm[1, 0]),
    })

tabla_final = pd.DataFrame(filas_final).set_index("Modelo")
tabla_final.round(4)

### 4.2 Matrices de confusión y reporte de clasificación

Fíjate en cómo se lee cada celda de la matriz con `0 = Maligno` y `1 = Benigno`:
- **Arriba izquierda:** Maligno real → predicho Maligno (acierto crítico).
- **Arriba derecha (`0 → 1`, Falso benigno):** Maligno real → predicho Benigno (**error más peligroso**, se deja ir un cáncer sin tratar).
- **Abajo izquierda (`1 → 0`, Falsa alarma):** Benigno real → predicho Maligno (molestia y prueba extra, pero no pone en riesgo la vida).
- **Abajo derecha:** Benigno real → predicho Benigno (acierto).

In [ ]:
modelos_cm = [
    "1. Base sin escalar (C=1.0)",
    "3. Estandarizado · Grid Acc (C=0.1)",
    "4. Estandarizado · Grid AUC (C=1.0)",
]
nombres_clases = ["Maligno (0)", "Benigno (1)"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, nombre in zip(axes, modelos_cm):
    pred = modelos[nombre].predict(X_test)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=nombres_clases,
        yticklabels=nombres_clases,
        ax=ax,
    )
    ax.set_title(f"{nombre}\nAcc={accuracy_score(y_test, pred):.3f} | Rec(Maligno)={recall_score(y_test, pred, pos_label=0):.3f}")
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")

plt.tight_layout()
plt.show()

modelo_ganador = modelos["4. Estandarizado · Grid AUC (C=1.0)"]
print("Reporte de clasificación · Modelo ganador (Estandarizado C=1.0):")
print(classification_report(y_test, modelo_ganador.predict(X_test), target_names=nombres_clases, digits=3))

El modelo **`Estandarizado · Grid AUC (C=1.0)`** acierta **41 de 42 tumores malignos** (`recall = 0,976`) y **71 de 72 benignos** (`recall = 0,986`), alcanzando **98,25 % de accuracy** y **0,981 de F1 macro** en prueba.

### 4.3 Curvas ROC y AUC en Test

Como la clínica busca detectar **tumores malignos (`0`)**, trazamos la curva ROC usando la probabilidad predicha de que una muestra sea maligna (`predict_proba(X_test)[:, 0]` frente a `y_test == 0`). En clasificación binaria el **AUC** es idéntico al calcularlo para la clase `1`, pero así el eje vertical representa directamente la **sensibilidad para detectar malignidad (TPR)**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
y_test_maligno = (y_test == 0).astype(int)
estilos = ["-", "--", "-.", ":"]

for (nombre, modelo), ls in zip(modelos.items(), estilos):
    prob_maligno = modelo.predict_proba(X_test)[:, 0]
    fpr, tpr, _ = roc_curve(y_test_maligno, prob_maligno)
    auc_val = roc_auc_score(y_test_maligno, prob_maligno)
    axes[0].plot(fpr, tpr, lw=2.2, ls=ls, alpha=0.9, label=f"{nombre} (AUC = {auc_val:.4f})")
    axes[1].plot(fpr, tpr, lw=2.4, ls=ls, alpha=0.9, label=f"{nombre} (AUC = {auc_val:.4f})")

axes[0].plot([0, 1], [0, 1], "k--", label="Azar (AUC = 0.5000)")
axes[0].set_title("Curvas ROC completas en Test (tumor maligno)")
axes[0].set_xlabel("Tasa de falsos positivos (FPR · falsas alarmas)")
axes[0].set_ylabel("Tasa de verdaderos positivos (TPR · recall maligno)")
axes[0].legend(loc="lower right", fontsize=8.5)

axes[1].set_xlim(-0.01, 0.25)
axes[1].set_ylim(0.85, 1.01)
axes[1].set_title("Zoom esquina superior izquierda (FPR ≤ 0,25)")
axes[1].set_xlabel("Tasa de falsos positivos (FPR · falsas alarmas)")
axes[1].set_ylabel("Tasa de verdaderos positivos (TPR · recall maligno)")
axes[1].legend(loc="lower right", fontsize=8.5)

plt.tight_layout()
plt.show()

Todos los modelos estandarizados alcanzan un **AUC superior a 0,995** en prueba (casi pegado a la esquina superior izquierda, `AUC = 1,0`), demostrando que la regresión logística ordena extraordinariamente bien los casos de menor a mayor riesgo.

### 4.4 Curva de aprendizaje e importancia de variables

Comprobamos si el modelo elegido (`Estandarizado C=1.0`) sufre sobreajuste observando su **curva de aprendizaje**, e inspeccionamos los **coeficientes** estandarizados (`coef_`) para saber qué variables pesan más en su decisión.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Izquierda: Curva de aprendizaje en validación cruzada (sobre train)
tamanos, sc_train, sc_val = learning_curve(
    modelo_ganador,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    train_sizes=np.linspace(0.2, 1.0, 5),
)
axes[0].plot(tamanos, sc_train.mean(axis=1), "o-", label="Train")
axes[0].fill_between(
    tamanos,
    sc_train.mean(axis=1) - sc_train.std(axis=1),
    sc_train.mean(axis=1) + sc_train.std(axis=1),
    alpha=0.15,
)
axes[0].plot(tamanos, sc_val.mean(axis=1), "s-", label="Validación (CV)")
axes[0].fill_between(
    tamanos,
    sc_val.mean(axis=1) - sc_val.std(axis=1),
    sc_val.mean(axis=1) + sc_val.std(axis=1),
    alpha=0.15,
)
axes[0].set_ylim(0.90, 1.01)
axes[0].set_title("Curva de aprendizaje (Estandarizado C=1.0)")
axes[0].set_xlabel("Número de muestras de entrenamiento")
axes[0].set_ylabel("Accuracy")
axes[0].legend(loc="lower right")

# Derecha: Top 12 coeficientes más influyentes (variables ya estandarizadas)
coefs = pd.Series(modelo_ganador.named_steps["logistica"].coef_[0], index=X.columns)
top_coefs = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(12)
df_coefs = pd.DataFrame({
    "Variable": top_coefs.index,
    "Coeficiente": top_coefs.values,
    "Efecto": ["Empuja hacia Maligno (0)" if c < 0 else "Empuja hacia Benigno (1)" for c in top_coefs.values],
})
sns.barplot(
    data=df_coefs,
    x="Coeficiente",
    y="Variable",
    hue="Efecto",
    palette={"Empuja hacia Maligno (0)": "#d95f02", "Empuja hacia Benigno (1)": "#1b9e77"},
    ax=axes[1],
)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_xlim(-1.35, 1.05)
axes[1].legend(title="Efecto", loc="upper right", fontsize=8.5)
axes[1].set_title("Top 12 coeficientes estandarizados (C=1.0)")

plt.tight_layout()
plt.show()

- **Sin sobreajuste:** en la curva de aprendizaje, entrenamiento (`0,989`) y validación (`0,978`) convergen muy rápido con apenas un **1,1 % de diferencia**.
- **Variables clave:** `worst texture` (`-1,26`), `radius error` (`-1,08`), `worst concave points` (`-0,95`), `worst area` (`-0,95`) y `worst radius` (`-0,95`) tienen los pesos negativos más fuertes, es decir, valores altos en ellas empujan la predicción hacia **Maligno (`0`)**.

### 4.5 Costo asimétrico de los errores: ajuste del umbral de decisión (`predict_proba`)

Por defecto, `predict()` solo marca un tumor como maligno si $P(\text{maligno}) \ge 0.50$. Pero **decir que un tumor maligno es benigno es mucho más grave que el error inverso**:
- Un **falso benigno (`0 → 1`)** significa mandar a casa sin tratamiento a alguien que sí tiene cáncer.
- Una **falsa alarma (`1 → 0`)** significa hacer una biopsia o estudio de confirmación a alguien sano.

Como la regresión logística devuelve probabilidades muy bien calibradas (`AUC ≈ 0,996`), podemos exigir **más cautela**: marcar como maligno cuando $P(\text{maligno})$ supere un umbral menor (por ejemplo, `0,30` en lugar de `0,50`). **Importante:** elegimos el umbral usando validación cruzada sobre `X_train` (`cross_val_predict`) para no hacer trampa mirando `X_test`, y después comprobamos qué ocurre en `X_test`.

In [ ]:
oof_prob_maligno = cross_val_predict(
    pipe_base.set_params(logistica__C=1.0),
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
)[:, 0]

umbrales = [0.50, 0.40, 0.30, 0.25, 0.20]
filas_umbral = []
for u in umbrales:
    pred_u = np.where(oof_prob_maligno >= u, 0, 1)
    cm_u = confusion_matrix(y_train, pred_u, labels=[0, 1])
    filas_umbral.append({
        "Umbral P(maligno) >=": u,
        "Acc CV": accuracy_score(y_train, pred_u),
        "Recall Maligno (0)": recall_score(y_train, pred_u, pos_label=0),
        "Recall Benigno (1)": recall_score(y_train, pred_u, pos_label=1),
        "Falsos benignos CV (0→1)": int(cm_u[0, 1]),
        "Falsas alarmas CV (1→0)": int(cm_u[1, 0]),
    })

tabla_umbral = pd.DataFrame(filas_umbral)
display(tabla_umbral.round(4))

# Aplicamos en Test el umbral 0.30 elegido en CV frente al estándar 0.50
prob_test_maligno = modelo_ganador.predict_proba(X_test)[:, 0]
pred_test_050 = np.where(prob_test_maligno >= 0.50, 0, 1)
pred_test_030 = np.where(prob_test_maligno >= 0.30, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (titulo, pred) in zip(
    axes,
    [
        ("Test con umbral estándar 0.50", pred_test_050),
        ("Test con umbral conservador 0.30", pred_test_030),
    ],
):
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=nombres_clases,
        yticklabels=nombres_clases,
        ax=ax,
    )
    ax.set_title(f"{titulo}\nAcc={accuracy_score(y_test, pred):.3f} | Rec(Maligno)={recall_score(y_test, pred, pos_label=0):.3f}")
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")

plt.tight_layout()
plt.show()

En las predicciones de validación cruzada sobre `train` (455 pacientes), bajar el umbral de alerta maligna de `0,50` a `0,30` **reduce a la mitad los falsos benignos** (de `6` a `3`, subiendo el recall de malignos de `96,5 %` a **`98,2 %`**) a cambio de aumentar las falsas alarmas de `4` a `10`. En `test` (114 pacientes), el modelo ya detectaba `41/42` malignos en `0,50`, y con `0,30` marca `4` pacientes benignos adicionales como sospechosos para revisión médica.

---
## 5. Conclusiones

1. **Qué reveló el EDA:**
   - El dataset tiene **569 muestras y 30 variables numéricas**, sin nulos ni duplicados.
   - Las clases están **desbalanceadas**: `357` benignos (`62,7 %`) y `212` malignos (`37,3 %`). Por eso la *accuracy* por sí sola engaña (predecir todo benigno ya daría `62,7 %`).
   - Las variables mejor correlacionadas y que más separan las dos clases son **`worst concave points` (`-0,79`)**, **`worst perimeter` (`-0,78`)**, **`mean concave points` (`-0,78`)** y **`worst radius` (`-0,78`)**. Los tumores malignos tienden a tener núcleos más grandes e irregulares.

2. **Decisión de preprocesamiento (`StandardScaler` dentro de un `Pipeline`):**
   - **Estandarizar fue imprescindible:** sin escalado (`worst area` hasta `4254` vs. `mean smoothness` ~`0,10`), la regresión logística ni siquiera converge en 1000 iteraciones (`ConvergenceWarning`) y su accuracy en CV cae a `0,9495` (con recall maligno de `0,9176`). Con `StandardScaler`, converge limpio y sube a `0,9780` de accuracy en CV y `0,9959` de AUC.
   - Probamos además `np.log1p` por la asimetría de 22 columnas: en CV logró resultados casi idénticos (`0,9958` de AUC y `0,9706` de recall maligno), pero **en test no superó** al escalado simple (`0,9737` vs. `0,9825` de accuracy, con el mismo `1` falso benigno), por lo que preferimos el pipeline más simple con `StandardScaler`.

3. **Qué modelo me quedo y por qué:**
   - Elegimos **`Pipeline([StandardScaler(), LogisticRegression(C=1.0, max_iter=1000, random_state=42)])`**.
   - Consiguió el **mejor AUC en validación cruzada (`0,9959`)**, un sólido **`0,9647` de recall en malignos en CV** y el **mejor resultado global en Test**: **`98,25 %` de accuracy**, **`0,9812` de F1 macro**, **`0,9954` de AUC** y **`97,62 %` de recall en malignos** (41 de 42 malignos detectados y 71 de 72 benignos bien clasificados).
   - Aunque `C = 0,1` obtuvo una accuracy ligeramente mayor en CV (`0,9824`), dejó escapar más tumores malignos tanto en CV (`recall = 0,9588`) como en test con el umbral estándar (`2` falsos benignos vs. `1` con `C = 1,0`).
   - Además, la curva de aprendizaje muestra poca diferencia entre entrenamiento y validación (`1,1 %`), confirmando que **no está sobreajustado**.

4. **Asimetría del error en el diagnóstico médico:**
   - **Decir que un tumor maligno es benigno (`0 → 1`, falso benigno)** es muchísimo más grave que decir que un benigno es maligno (`1 → 0`, falsa alarma): el primero retrasa el tratamiento de un cáncer, mientras que el segundo se corrige con una prueba confirmatoria.
   - Gracias a que `LogisticRegression` entrega probabilidades bien calibradas (`predict_proba`), en un entorno clínico real conviene acompañar este modelo con un **umbral más prudente** (por ejemplo, marcar sospecha si $P(\text{maligno}) \ge 0,30$ en lugar de `0,50`, lo que en validación cruzada redujo los falsos benignos a la mitad).